<a href="https://colab.research.google.com/github/amallindask/data-science-2026/blob/main/Pertemuan10_Amallinda_250401020003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IDENTITAS** **MAHASISWA**

---

Nama : Amallinda Sekar Kinasih

---

NIM : 25040102003

---
Kelas : IF403


In [6]:
# ==========================================
# CUSTOMER CHURN PREDICTION
# ==========================================

# Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)


In [15]:
from google.colab import drive
drive.mount('/content/drive')

!find /content/drive/MyDrive -name "*WA_Fn-UseC_-Telco-Customer-Churn.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/archive/WA_Fn-UseC_-Telco-Customer-Churn.csv
/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [39]:
# ==========================================
# Langkah 1 : Muat dan Eksplorasi Data
# ==========================================

#df = pd.read_csv("telco_churn.csv")
df = pd.read_csv('/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("Ukuran Dataset:")
print(df.shape)

print("\n5 Data Pertama:")
print(df.head())

print("\nInformasi Dataset:")
print(df.info())

print("\nProporsi Kelas Churn:")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset:
(7043, 21)

5 Data Pertama:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Stre

In [40]:
# ==========================================
# Langkah 2 : Preprocessing
# ==========================================

# Menghapus customerID karena hanya ID pelanggan
if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)

# Mengubah TotalCharges menjadi numerik
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

    df["TotalCharges"] = df["TotalCharges"].fillna(
        df["TotalCharges"].median()
    )
# Mengubah target menjadi numerik
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# One-Hot Encoding
df_encoded = pd.get_dummies(df, drop_first=True)

# Pisahkan fitur dan target
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

from sklearn.model_selection import train_test_split

print(df["Churn"].isna().sum())

0


In [41]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Ukuran Data Latih:", X_tr.shape)
print("Ukuran Data Uji:", X_te.shape)

Ukuran Data Latih: (5634, 30)
Ukuran Data Uji: (1409, 30)


In [42]:
# ==========================================
# Langkah 3 : Latih Model
# ==========================================

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced",
    random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [43]:
# ==========================================
# Langkah 4 : Evaluasi
# ==========================================

y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]

print("\nConfusion Matrix")
print(confusion_matrix(y_te, y_pred))

print("\nClassification Report")
print(classification_report(y_te, y_pred))

roc_auc = roc_auc_score(y_te, y_prob)
print("ROC-AUC Score :", round(roc_auc, 4))


Confusion Matrix
[[925 110]
 [187 187]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score : 0.8246


In [45]:
# ==========================================
# Langkah 5 : Prediksi Probabilitas Churn
# ==========================================

hasil = pd.DataFrame({
    "Probabilitas_Churn": y_prob,
    "Prediksi_Churn": y_pred
})

print("\n10 Pelanggan Pertama:")
print(hasil.head(10))

# Menampilkan pelanggan dengan risiko churn tertinggi
print("\n\n10 Pelanggan Risiko Churn Tertinggi:")
print(
    hasil.sort_values(
        by="Probabilitas_Churn",
        ascending=False
    ).head(10)
)


10 Pelanggan Pertama:
   Probabilitas_Churn  Prediksi_Churn
0            0.000000               0
1            0.786667               1
2            0.090000               0
3            0.280000               0
4            0.000000               0
5            0.416667               0
6            0.393333               0
7            0.110000               0
8            0.006667               0
9            0.460000               0


10 Pelanggan Risiko Churn Tertinggi:
      Probabilitas_Churn  Prediksi_Churn
1289            1.000000               1
171             0.993333               1
341             0.990000               1
618             0.990000               1
1252            0.986667               1
629             0.986667               1
889             0.970000               1
1178            0.963333               1
1109            0.960000               1
788             0.950000               1


## Kesimpulan
### Berdasarkan hasil pengujian, model Random Forest berhasil digunakan untuk memprediksi customer churn pada dataset Telco Customer Churn. Penggunaan parameter class_weight='balanced' membantu model dalam menangani ketidakseimbangan jumlah data antara pelanggan yang churn dan tidak churn. Hasil evaluasi menunjukkan bahwa model memiliki kemampuan yang baik dalam mengklasifikasikan pelanggan berdasarkan kemungkinan mereka berhenti menggunakan layanan, yang ditunjukkan oleh nilai precision, recall, F1-score, dan ROC-AUC yang diperoleh. Selain itu, probabilitas churn yang dihasilkan melalui metode predict_proba() dapat dimanfaatkan perusahaan untuk mengidentifikasi pelanggan yang berisiko tinggi melakukan churn sehingga strategi retensi dapat dilakukan secara lebih efektif dan tepat sasaran.